# BoW 기반 텍스트 분석 연습
- **Google Play 듀오링고(Duolingo) 리뷰 데이터셋 크롤링**
    - 언어 학습 어플 '듀오링고'의 리뷰 데이터를 크롤링하여 리뷰를 분석한다.

1. 적절한 데이터셋을 찾거나 생성하고, 적절한 전처리를 진행한다. (01_preprocessing)
2. TF-IDF Vectorizer를 이용하여 벡터화
3. Cosine Simillarity를 계산하여 입력된 문자열의 긍/부정 판단

vectorizer를 이용해서 vector -> 코사인 유사도 + 클러스터링(가까이 있는 걸 묶어줌)

# 1. 데이터 크롤링

In [80]:
# !pip install google-play-scraper

In [81]:
# !pip install kiwipiepy

### 1-1. 리뷰 50000개 추출

In [82]:
import pandas as pd

In [83]:
from google_play_scraper import reviews, Sort

result, continuation_token = reviews(
    'com.duolingo', # 앱 ID
    lang='ko',         # 언어
    country='kr',      # 국가
    sort=Sort.NEWEST,  # 최신순
    count=50000,        # 리뷰 수
    filter_score_with=None # 평점 필터링 (1~5)
)

### 1-2. 결과 확인 후 평점별 리뷰 개수 조정

In [84]:
df = pd.DataFrame(result)
df.count()

reviewId                50000
userName                50000
userImage               50000
content                 49998
score                   50000
thumbsUpCount           50000
reviewCreatedVersion    47553
at                      50000
replyContent                0
repliedAt                   0
appVersion              47553
dtype: int64

# 2. 전처리

### 2-1. 기본적인 전처리 과정

In [85]:
df = df[df['content'].str.replace(" ", "").str.len() > 10]

In [86]:
# 1. 한글만 포함된 것 거르기
import re

df = df[df['content'].str.contains('[가-힣]', regex=True, na=False)]
df.count()

reviewId                31620
userName                31620
userImage               31620
content                 31620
score                   31620
thumbsUpCount           31620
reviewCreatedVersion    29869
at                      31620
replyContent                0
repliedAt                   0
appVersion              29869
dtype: int64

In [87]:
# 2. 결측치가 있는 reviewCreatedVersion, appVersion에 맞춰서 숫자를 맞춰줌
df = df.dropna(subset=['reviewCreatedVersion', 'appVersion'])
df.count()

reviewId                29869
userName                29869
userImage               29869
content                 29869
score                   29869
thumbsUpCount           29869
reviewCreatedVersion    29869
at                      29869
replyContent                0
repliedAt                   0
appVersion              29869
dtype: int64

In [88]:
import re
from kiwipiepy import Kiwi

kiwi = Kiwi()

def tokenize(text):
    text = str(text)

    # 1. 이모지 제거
    text = re.sub(r'[\U00010000-\U0010ffff\u2600-\u27ff\u200d\ufe0f]', ' ', text)

    # 2. 숫자+단위 붙이기 (토큰화 전)
    text = re.sub(r'(\d+)\s+(주차|주|학년|개월|년|일|시간|분)', r'\1\2', text)

    tokens = []

    for token in kiwi.tokenize(text):
        form = token.form
        tag = token.tag

        # 3. 조사/어미 제거
        if tag.startswith('J') or tag.startswith('E'):
            continue

        # 4. 동사/형용사/보조용언 계열 원형화
        if tag.startswith('V') or tag in ['XSV', 'XSA']:
            word = form + '다'
        else:
            word = form

        # 5. 자모 제거
        if re.fullmatch(r'[ㄱ-ㅎㅏ-ㅣ]+', word):
            continue
        if word in {'ㅇ', 'ㄴ', 'ㄹ', 'ㅁ', 'ㅋ', 'ㅎ', 'ㅠ', 'ㅜ'}:
            continue

        # 6. 공백 제거
        word = word.strip()
        if not word:
            continue

        if re.fullmatch(r'[!?.,]+', word):
            continue

        tokens.append(word)

    # 7. 토큰 후처리: 숫자 + 단위 다시 합치기
    merged = []
    i = 0
    while i < len(tokens):
        if i + 1 < len(tokens) and tokens[i].isdigit():
            # 2 + 주차
            if tokens[i+1] in {'주차', '주', '학년', '개월', '년', '일', '시간', '분'}:
                merged.append(tokens[i] + tokens[i+1])
                i += 2
                continue
            # 2 + 주 + 차
            if i + 2 < len(tokens) and tokens[i+1] == '주' and tokens[i+2] == '차':
                merged.append(tokens[i] + '주차')
                i += 3
                continue

        merged.append(tokens[i])
        i += 1

    return merged

In [89]:
df['tokens'] = df['content'].apply(tokenize)

In [90]:
df.head()

,reviewId,userName,userImage,content,score,thumbsUpCount,reviewCreatedVersion,at,replyContent,repliedAt,appVersion,tokens
1,ebad105f-706a-4db5-8945-35615ef173b2,Google 사용자,https://play-lh.googleusercontent.com/EGemoI2N...,다른건 다 좋은데 음성인식에 찐빠가 있네요...,4,0,6.69.3,2026-03-07 19:23:51,None,None,6.69.3,"[다른, 거, 다, 좋다, 음성, 인식, 찐빠, 있다]"
2,7009c886-b768-4c92-9a32-3a5e7e0c99f2,Google 사용자,https://play-lh.googleusercontent.com/EGemoI2N...,지금 결제된거 취소좀 해주세요 2일전에 알려준다더니 왜 안알려주고 그냥 결제하게 하나요?,1,0,6.69.3,2026-03-07 18:16:01,None,None,6.69.3,"[지금, 결제, 되다, 거, 취소, 좀, 하다, 주다, 2일, 전, 알리다, 주다,..."
3,c52cead6-e0f1-400c-8206-dba2edf79154,Google 사용자,https://play-lh.googleusercontent.com/EGemoI2N...,잇츠소굿 말해뭐해✨️✨️,5,0,6.69.3,2026-03-07 18:04:09,None,None,6.69.3,"[잇츠소굿, 말, 하다, 뭐, 하다]"
4,8f49d9f5-7605-4f1d-a52b-6ed20c73681f,Google 사용자,https://play-lh.googleusercontent.com/EGemoI2N...,이거하고 나서 끈기있다는말 처음으로 들어봣어요!!,5,0,6.69.3,2026-03-07 17:45:29,None,None,6.69.3,"[이거, 하다, 나다, 끈기, 있다, 말, 처음, 듣다, 보다]"
7,14b3b43f-d635-4e97-a4ca-2af7a14a762f,Google 사용자,https://play-lh.googleusercontent.com/EGemoI2N...,잘 되다가 한번씩 앱이 멈추고 응답하지않음이 떠요 고쳐주세요 그거 말곤 항상 잘 사...,5,0,6.69.3,2026-03-07 16:24:49,None,None,6.69.3,"[잘, 되다, 한, 번, 씩, 앱, 멈추다, 응답, 하다, 않다, 뜨다, 고치다, ..."


### 2-2. 각 리뷰별로 데이터 개수 맞추기

In [91]:
df['score'].value_counts()

score
5    19538
4     4860
1     2501
3     2058
2      912
Name: count, dtype: int64

In [92]:
# 각각 900개씩 맞춰서 총 2000개 맞추기
df_duolingo = (
   df.groupby('score', group_keys=False)
    .apply(lambda x: x.sample(n=900, random_state=42))
    .reset_index(drop=True)
)

C:\Users\yunu2\AppData\Local\Temp\ipykernel_36256\1406339057.py:4: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(n=900, random_state=42))


In [93]:
df_duolingo['score'].value_counts()

score
1    900
2    900
3    900
4    900
5    900
Name: count, dtype: int64

In [94]:
df_duolingo.count()

reviewId                4500
userName                4500
userImage               4500
content                 4500
score                   4500
thumbsUpCount           4500
reviewCreatedVersion    4500
at                      4500
replyContent               0
repliedAt                  0
appVersion              4500
tokens                  4500
dtype: int64

### 2-2. 필요한 열만 추출
- reviewid, score, content, tokens

In [95]:
df_duolingo = df_duolingo[['reviewId', 'score','content', 'tokens']]
df_duolingo.head()

,reviewId,score,content,tokens
0,2dd6b1c2-fcf4-4e99-a743-5fa880d6004a,1,저렴하게 언어공부 가볍게 하기엔 그냥저냥인데 캐릭터들이 참 4가지없네요. 스토리도 ...,"[저렴하다, 언어, 공부, 가볍다, 하다, 그냥저냥, 이다, 캐릭터, 들, 참, 4..."
1,513f5df6-ac02-4a4e-bff2-130c8d470b52,1,최근 버전 업데이트 이후 너무 불편해요. 학습을 제대로 할 수가 없어요. 2년 이상...,"[최근, 버전, 업데이트, 이후, 너무, 불편, 하다, 학습, 제대로, 하다, 수,..."
2,7ea1725f-ee3d-4a1e-ad01-132529e41742,1,광고너무뜨네요 원래 안뜨거나 1개였던거 같은데 이제 계속 2개씩 떠서 앱을 사용하는...,"[광고, 너무, 뜨다, 원래, 안, 뜨다, 1, 개, 이다, 거, 같다, 이제, 계..."
3,4adc04ff-3f84-4b32-ba55-b574d4701151,1,스피킹시 인식을잘못함 하트만날림,"[스피킹, 시, 인식, 잘, 못, 하다, 하트, 날리다]"
4,fd67cb04-2b7a-4d69-9d1c-cf3453fc5fa0,1,도대체 왜 하트를 에너지로 만들고 맞쳐도 1까이고 틀려도 까이는건 공부하지 말라는거...,"[도대체, 왜, 하트, 에너지, 만들다, 맞다, 치다, 1, 이다, 틀리다, 까다,..."


# 3. TF-IDF를 이용한 벡터화

In [97]:
from sklearn.feature_extraction.text import TfidfVectorizer

df_duolingo['tokens_str'] = df_duolingo['tokens'].apply(lambda x: ' '.join(x))

tfidf_vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    max_df=0.85,
    min_df=3
)

opinion_vecs = tfidf_vectorizer.fit_transform(df_duolingo['tokens_str'])

# 4. Cosine Similarity를 이용한 긍/부정 판단

In [98]:
from sklearn.metrics.pairwise import cosine_similarity

cos_sim = cosine_similarity(opinion_vecs, opinion_vecs)
cos_sim.shape

(4500, 4500)

### 4-1. 평점 기준으로 집단 만들기

In [99]:
def label_score(x):
    if x >= 4:
        return 1      # positive
    elif x <= 2:
        return -1     # negative
    else:
        return 0      # neutral

df_duolingo['polarity'] = df_duolingo['score'].apply(label_score)

### 4-2. 긍정/부정 벡터 만들기

In [100]:
import numpy as np

pos_idx = df_duolingo[df_duolingo['polarity'] == 1].index
neu_idx = df_duolingo[df_duolingo['polarity'] == 0].index
neg_idx = df_duolingo[df_duolingo['polarity'] == -1].index

pos_centroid = np.asarray(opinion_vecs[pos_idx].mean(axis=0)).reshape(1,-1)
neu_centroid = np.asarray(opinion_vecs[neu_idx].mean(axis=0)).reshape(1,-1)
neg_centroid = np.asarray(opinion_vecs[neg_idx].mean(axis=0)).reshape(1,-1)

### 4-3. 각 리뷰와 긍정, 부정 중심의 코사인 유사도 계산

In [101]:
from sklearn.metrics.pairwise import cosine_similarity

pos_sim = cosine_similarity(opinion_vecs, pos_centroid).flatten()
neu_sim = cosine_similarity(opinion_vecs, neu_centroid).flatten()
neg_sim = cosine_similarity(opinion_vecs, neg_centroid).flatten()

In [102]:
df_duolingo['pos_sim'] = pos_sim
df_duolingo['neu_sim'] = neu_sim
df_duolingo['neg_sim'] = neg_sim

In [103]:
import numpy as np

sim_matrix = np.vstack([neg_sim, neu_sim, pos_sim]).T
labels = np.argmax(sim_matrix, axis=1) - 1

df_duolingo['pred_polarity'] = labels

In [104]:
df_duolingo.head()

,reviewId,score,content,tokens,tokens_str,polarity,pos_sim,neu_sim,neg_sim,pred_polarity
0,2dd6b1c2-fcf4-4e99-a743-5fa880d6004a,1,저렴하게 언어공부 가볍게 하기엔 그냥저냥인데 캐릭터들이 참 4가지없네요. 스토리도 ...,"[저렴하다, 언어, 공부, 가볍다, 하다, 그냥저냥, 이다, 캐릭터, 들, 참, 4...",저렴하다 언어 공부 가볍다 하다 그냥저냥 이다 캐릭터 들 참 4 가지 없다 스토리 ...,-1,0.161748,0.209441,0.220928,-1
1,513f5df6-ac02-4a4e-bff2-130c8d470b52,1,최근 버전 업데이트 이후 너무 불편해요. 학습을 제대로 할 수가 없어요. 2년 이상...,"[최근, 버전, 업데이트, 이후, 너무, 불편, 하다, 학습, 제대로, 하다, 수,...",최근 버전 업데이트 이후 너무 불편 하다 학습 제대로 하다 수 없다 2년 이상 하다...,-1,0.274977,0.307943,0.327190,-1
2,7ea1725f-ee3d-4a1e-ad01-132529e41742,1,광고너무뜨네요 원래 안뜨거나 1개였던거 같은데 이제 계속 2개씩 떠서 앱을 사용하는...,"[광고, 너무, 뜨다, 원래, 안, 뜨다, 1, 개, 이다, 거, 같다, 이제, 계...",광고 너무 뜨다 원래 안 뜨다 1 개 이다 거 같다 이제 계속 2 개 씩 뜨다 앱 ...,-1,0.132244,0.193762,0.213229,-1
3,4adc04ff-3f84-4b32-ba55-b574d4701151,1,스피킹시 인식을잘못함 하트만날림,"[스피킹, 시, 인식, 잘, 못, 하다, 하트, 날리다]",스피킹 시 인식 잘 못 하다 하트 날리다,-1,0.052863,0.091905,0.106344,-1
4,fd67cb04-2b7a-4d69-9d1c-cf3453fc5fa0,1,도대체 왜 하트를 에너지로 만들고 맞쳐도 1까이고 틀려도 까이는건 공부하지 말라는거...,"[도대체, 왜, 하트, 에너지, 만들다, 맞다, 치다, 1, 이다, 틀리다, 까다,...",도대체 왜 하트 에너지 만들다 맞다 치다 1 이다 틀리다 까다 이다 거 공부 하다 ...,-1,0.162678,0.203894,0.237937,-1


# 5. 열 이름 정리

In [105]:
df_duolingo.columns = ['id', 'rate', 'comment', 'comment_tokens_list', 'comment_tokens', 'polarity', 'pos_cos_sim', 'neu_cos_sim', 'neg_cos_sim', 'pred_polarity']

In [106]:
import csv

df_save = df_duolingo.copy()

if 'tokens' in df_save.columns:
    df_save['tokens'] = df_save['tokens'].apply(
        lambda x: ' '.join(x) if isinstance(x, list) else str(x)
    )

df_save.to_csv(
    'result_for_duolingo_text_anal.csv',
    index=False,
    encoding='utf-8-sig',
    escapechar='\\'
)